In [3]:
import pandas as pd

# 1. Carregar o conjunto de dados do compressor de ar a partir de um ficheiro CSV
df = pd.read_csv("MetroPT3(AirCompressor).csv")

# 2. Remover a coluna "Unnamed: 0" que geralmente surge do índice guardado no CSV
df = df.drop(columns=["Unnamed: 0"])

# 3. Converter a coluna "timestamp" de texto (string) para o formato de data e hora (datetime) do Pandas
df["timestamp"] = pd.to_datetime(df["timestamp"])

# 4. Ordenar os dados por ordem cronológica (do mais antigo para o mais recente) e reiniciar os índices
df = df.sort_values("timestamp").reset_index(drop=True)

# 5. Imprimir estatísticas básicas e inspeção inicial dos dados
print("Dimensão:", df.shape)                                    # Mostra o número de linhas e colunas
print("Período:", df["timestamp"].min(), "->", df["timestamp"].max())  # Mostra a data inicial e final do dataset
print("Tipo do timestamp:", df["timestamp"].dtype)              # Confirma se o tipo de dados do timestamp está correto
print(df.dtypes)                                                # Mostra o tipo de dados de todas as colunas
print("Valores em falta:", df.isna().sum().sum())               # Conta o total de valores nulos (NaN) em todo o DataFrame
print(df.describe().T)                                          # Mostra estatísticas descritivas (média, min, max, etc.) transpostas

Dimensão: (1516948, 16)
Período: 2020-02-01 00:00:00 -> 2020-09-01 03:59:50
Tipo do timestamp: datetime64[us]
timestamp          datetime64[us]
TP2                       float64
TP3                       float64
H1                        float64
DV_pressure               float64
Reservoirs                float64
Oil_temperature           float64
Motor_current             float64
COMP                      float64
DV_eletric                float64
Towers                    float64
MPG                       float64
LPS                       float64
Pressure_switch           float64
Oil_level                 float64
Caudal_impulses           float64
dtype: object
Valores em falta: 0
                     count                        mean                  min  \
timestamp          1516948  2020-05-16 22:58:36.631517  2020-02-01 00:00:00   
TP2              1516948.0                    1.367826               -0.032   
TP3              1516948.0                    8.984611                 0.73

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# 6. Definir intervalos de tempo específicos em que ocorreram falhas conhecidas no compressor
falhas = [
    ("2020-04-18 00:00", "2020-04-18 23:59"),
    ("2020-05-29 23:30", "2020-05-30 06:00"),
    ("2020-06-05 10:00", "2020-06-07 14:30"),
    ("2020-07-15 14:30", "2020-07-15 19:00"),
]

# 7. Criar a coluna alvo "falha" e inicializá-la a 0 (estado normal) para todas as linhas
df["falha"] = 0

# 8. Percorrer os intervalos de falha definidos e alterar o valor da coluna "falha" para 1 (anomalia) nesses períodos
for inicio, fim in falhas:
    df.loc[df["timestamp"].between(inicio, fim), "falha"] = 1

# 9. Mostrar a distribuição das classes (quantas linhas normais vs. quantas falhas existem)
print("Distribuição das classes:")
print(df["falha"].value_counts())                  # Contagem absoluta
print(df["falha"].value_counts(normalize=True) * 100) # Percentagem de cada classe

# ==========================================
# ADIÇÃO: 10. Normalização do Dataset (Min-Max)
# ==========================================

# Identificar as colunas numéricas que representam os sensores (excluindo timestamp e a alvo 'falha')
colunas_sensores = df.select_dtypes(include=["float64", "int64"]).columns.drop("falha")

# Criar o objeto scaler para colocar os dados entre o intervalo 0 e 1
scaler = MinMaxScaler()

# Aplicar a normalização e substituir os valores originais no DataFrame pelas versões normalizadas
df[colunas_sensores] = scaler.fit_transform(df[colunas_sensores])

print("\nDataset normalizado com sucesso! Primeiras linhas das colunas dos sensores:")
print(df[colunas_sensores].head())

falha
0    1486994
1      29954
Name: count, dtype: int64
falha
0    98.025377
1     1.974623
Name: proportion, dtype: float64


In [6]:
# 10. Definir o ponto de corte para dividir os dados a 50% (metade mais antiga, metade mais recente)
corte = int(len(df) * 0.5)

# 11. Dividir o DataFrame cronologicamente em treino e teste
treino = df.iloc[:corte]  # Primeira metade para treino
teste = df.iloc[corte:]   # Segunda metade para teste

# 12. Separar as variáveis independentes (features/sensores) da variável alvo (target/falha)
# Removemos "timestamp" e "falha" das features porque não devem ser usadas diretamente como dados de entrada para prever a falha
X_train = treino.drop(columns=["timestamp", "falha"])
y_train = treino["falha"]

X_test = teste.drop(columns=["timestamp", "falha"])
y_test = teste["falha"]

# 13. Imprimir o resumo final dos conjuntos criados e o número de falhas em cada um
print("Treino:", len(X_train), "| falhas:", y_train.sum())
print("Teste: ", len(X_test), "| falhas:", y_test.sum())

Treino: 758474 | falhas: 8657
Teste:  758474 | falhas: 21297
